# 🧠 Healthcare RAG-Powered Medical QA Assistant

This notebook implements a **Retrieval-Augmented Generation (RAG)** pipeline
for answering medical questions using PubMedQA data.

### 🔍 What this notebook does:
- Loads and processes medical QA data
- Converts text into embeddings
- Stores embeddings using FAISS
- Retrieves relevant context
- Generates answers using an LLM

---

## ⚙️ Environment Setup

We import all required libraries for:
- Data processing
- Embeddings
- Vector search (FAISS)
- LLM interaction

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import faiss
import pickle

## 📊 Load Precomputed Data

We load:
- FAISS index (vector database)
- Chunk mapping (text data مرتبط بالـ embeddings)

In [2]:
index = faiss.read_index("data/embeddings/faiss_index/pubmedqa_index_flatl2.faiss")

with open("data/embeddings/faiss_index/chunk_mapping.pkl", "rb") as f:
    text_chunks = pickle.load(f)

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

d:\Projects\Healthcare-RAG-Powered-Medical-QA-Assistant\venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
d:\Projects\Healthcare-RAG-Powered-Medical-QA-Assistant\venv\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## 🔎 Retrieval

This step:
1. Converts the query into an embedding
2. Searches FAISS for similar vectors
3. Returns the most relevant text chunks

In [4]:
def retrieve(query, k=3):
    query_vector = model.encode([query])
    D, I = index.search(query_vector, k)
    return [text_chunks['question'].iloc[i] for i in I[0]]

In [5]:
query = "What are the symptoms of diabetes?"

results = retrieve(query)

context = "\n".join(results)

print("Context:\n", context)

Context:
 Abnormalities in glucose homeostasis in acromegaly. Does the prevalence of glucose intolerance depend on the level of activity of the disease and the duration of the symptoms?
Difficult diagnosis in a 17-year-old patient: Type 1 diabetes?
The course of diabetes in children, adolescents and young adults: does the autoimmunity status matter?


## 🔑 API Configuration

We securely load API keys from environment variables.

In [6]:
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

In [7]:
client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "system", "content": "You are a helpful medical assistant. Answer only based on the given context."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
    ]
)

print(response.choices[0].message.content)

Diabetes is a complex condition, and its symptoms can be quite nonspecific, especially in the initial stages. However, I'll outline the common symptoms of diabetes, which can include:

For Type 1 Diabetes:
- Increased thirst and dry mouth
- Frequent urination
- Fatigue
- Blurred vision
- Slow healing of cuts and wounds
- Tingling or numbness in hands and feet

For Type 2 Diabetes (as it relates to acromegaly):
- Similar symptoms to Type 1, but they can also include:
  - Polyphagia (increased hunger)
  - Polyuria (increased urination)
  - Weight gain

As we discussed in the context of acromegaly, glucose intolerance can arise due to the disorder's impact on growth hormone production, leading to insulin resistance.

In the case of a 17-year-old patient, determining whether they have Type 1 diabetes can be challenging, as the symptoms can be similar to those of other conditions.

The course of diabetes in children, adolescents, and young adults does matter, including the autoimmunity stat

In [8]:
DISCLAIMER = "\n\n⚠️ This system is for educational purposes only."

answer = response.choices[0].message.content + DISCLAIMER
print(answer)

Diabetes is a complex condition, and its symptoms can be quite nonspecific, especially in the initial stages. However, I'll outline the common symptoms of diabetes, which can include:

For Type 1 Diabetes:
- Increased thirst and dry mouth
- Frequent urination
- Fatigue
- Blurred vision
- Slow healing of cuts and wounds
- Tingling or numbness in hands and feet

For Type 2 Diabetes (as it relates to acromegaly):
- Similar symptoms to Type 1, but they can also include:
  - Polyphagia (increased hunger)
  - Polyuria (increased urination)
  - Weight gain

As we discussed in the context of acromegaly, glucose intolerance can arise due to the disorder's impact on growth hormone production, leading to insulin resistance.

In the case of a 17-year-old patient, determining whether they have Type 1 diabetes can be challenging, as the symptoms can be similar to those of other conditions.

The course of diabetes in children, adolescents, and young adults does matter, including the autoimmunity stat

In [9]:
log = {
    "question": query,
    "context": context,
    "answer": answer
}

print(log)

{'question': 'What are the symptoms of diabetes?', 'context': 'Abnormalities in glucose homeostasis in acromegaly. Does the prevalence of glucose intolerance depend on the level of activity of the disease and the duration of the symptoms?\nDifficult diagnosis in a 17-year-old patient: Type 1 diabetes?\nThe course of diabetes in children, adolescents and young adults: does the autoimmunity status matter?', 'answer': "Diabetes is a complex condition, and its symptoms can be quite nonspecific, especially in the initial stages. However, I'll outline the common symptoms of diabetes, which can include:\n\nFor Type 1 Diabetes:\n- Increased thirst and dry mouth\n- Frequent urination\n- Fatigue\n- Blurred vision\n- Slow healing of cuts and wounds\n- Tingling or numbness in hands and feet\n\nFor Type 2 Diabetes (as it relates to acromegaly):\n- Similar symptoms to Type 1, but they can also include:\n  - Polyphagia (increased hunger)\n  - Polyuria (increased urination)\n  - Weight gain\n\nAs we d